# Project 2 — Revenue Intelligence & GTM Analytics
## Step 1: Data Exploration & Validation

**Goal:** Before building dbt models or analysis scripts, understand what each dataset looks like, where the lifecycle chain breaks, and what the key data quality issues are.

**Datasets:**
| File | Description |
|------|-------------|
| `raw_salesforce_accounts.csv` | Company accounts (CRM master) |
| `raw_marketing_leads.csv` | Leads from all channels |
| `raw_salesforce_opportunities.csv` | Sales pipeline & closed deals |
| `raw_contracts.csv` | Signed contracts from Closed-Won opps |
| `raw_billing.csv` | Monthly invoices from contracts |
| `raw_product_events.csv` | Product usage events |
| `raw_customer_success.csv` | Health scores, renewals, churn |

---

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)
pd.set_option('display.max_colwidth', 40)

DATA_DIR = Path('../data/raw')

accounts = pd.read_csv(DATA_DIR / 'raw_salesforce_accounts.csv')
leads    = pd.read_csv(DATA_DIR / 'raw_marketing_leads.csv')
opps     = pd.read_csv(DATA_DIR / 'raw_salesforce_opportunities.csv')
contracts= pd.read_csv(DATA_DIR / 'raw_contracts.csv')
billing  = pd.read_csv(DATA_DIR / 'raw_billing.csv')
product  = pd.read_csv(DATA_DIR / 'raw_product_events.csv')
cs       = pd.read_csv(DATA_DIR / 'raw_customer_success.csv')

# Parse dates
accounts['account_created_at'] = pd.to_datetime(accounts['account_created_at'])
leads['created_at']            = pd.to_datetime(leads['created_at'])
opps['created_at']             = pd.to_datetime(opps['created_at'])
opps['expected_close_date']    = pd.to_datetime(opps['expected_close_date'])
opps['actual_close_date']      = pd.to_datetime(opps['actual_close_date'])
opps['last_activity_at']       = pd.to_datetime(opps['last_activity_at'])
contracts['contract_date']     = pd.to_datetime(contracts['contract_date'])
contracts['start_date']        = pd.to_datetime(contracts['start_date'])
contracts['end_date']          = pd.to_datetime(contracts['end_date'])
billing['billing_date']        = pd.to_datetime(billing['billing_date'])

print('✓ All datasets loaded successfully')

✓ All datasets loaded successfully


---
## 1. Dataset Overview
Row counts, column counts, and date ranges for every source table.

In [2]:
overview = pd.DataFrame([
    {'Dataset': 'accounts',     'Rows': len(accounts),  'Cols': len(accounts.columns),
     'Date Range': f"{accounts['account_created_at'].min().date()} → {accounts['account_created_at'].max().date()}"},
    {'Dataset': 'leads',        'Rows': len(leads),     'Cols': len(leads.columns),
     'Date Range': f"{leads['created_at'].min().date()} → {leads['created_at'].max().date()}"},
    {'Dataset': 'opportunities','Rows': len(opps),      'Cols': len(opps.columns),
     'Date Range': f"{opps['created_at'].min().date()} → {opps['created_at'].max().date()}"},
    {'Dataset': 'contracts',    'Rows': len(contracts), 'Cols': len(contracts.columns),
     'Date Range': f"{contracts['contract_date'].min().date()} → {contracts['contract_date'].max().date()}"},
    {'Dataset': 'billing',      'Rows': len(billing),   'Cols': len(billing.columns),
     'Date Range': f"{billing['billing_date'].min().date()} → {billing['billing_date'].max().date()}"},
    {'Dataset': 'product',      'Rows': len(product),   'Cols': len(product.columns),  'Date Range': '2024-01-01 → 2025-12-31'},
    {'Dataset': 'customer_success', 'Rows': len(cs),    'Cols': len(cs.columns),       'Date Range': '—'},
])

overview.style.set_properties(**{'text-align': 'left'}).hide(axis='index')

Dataset,Rows,Cols,Date Range
accounts,1212,9,2024-01-01 → 2025-12-31
leads,3200,9,2024-01-01 → 2025-12-31
opportunities,1800,14,2024-01-01 → 2025-10-30
contracts,377,9,2024-02-02 → 2026-03-23
billing,1725,7,2024-02-12 → 2026-01-30
product,25000,7,2024-01-01 → 2025-12-31
customer_success,697,6,—


### Quick preview of each table

In [3]:
print('── Accounts ──')
display(accounts.head(3))

print('\n── Opportunities ──')
display(opps.head(3))

print('\n── Contracts ──')
display(contracts.head(3))

print('\n── Billing ──')
display(billing.head(3))

── Accounts ──


,account_id,account_name,company_size,segment,industry,sales_region,revenue_model,account_created_at,account_status
0,ACC_00001,Cobalt Partners 0001,100-999,Mid-Market,Media,North America,Subscription,2024-01-05,Customer
1,ACC_00002,Cobalt Networks 0002,1-99,SMB,Financial Services,North America,Subscription,2025-02-18,Customer
2,ACC_00003,Atlas Labs 0003,1000+,Enterprise,Software,North America,Advertising,2025-06-29,Prospect



── Opportunities ──


,opportunity_id,account_id,owner_id,opportunity_stage,amount,expected_close_date,actual_close_date,created_at,sales_region,segment,revenue_model,lead_source,close_date_changes,last_activity_at
0,OPP_000001,ACC_00655,REP_015,Proposal,"40,662.83",2025-10-18,NaT,2025-09-20,LATAM,Enterprise,Advertising,Inbound,0,2025-11-10
1,OPP_000002,ACC_00580,REP_004,Proposal,"7,247.32",2025-07-29,NaT,2025-03-02,APAC,SMB,Enterprise Services,Marketing,0,2025-03-25
2,OPP_000003,ACC_00418,REP_011,Closed-Won,"18,778.17",2025-01-03,2025-02-07,2024-11-10,North America,Mid-Market,Subscription,Inbound,0,2024-11-15



── Contracts ──


,contract_id,account_id,opportunity_id,contract_date,contract_value,start_date,end_date,revenue_model,contract_status
0,CON_000001,ACC_00418,OPP_000003,2025-02-07,"19,654.55",2025-02-18,2026-02-13,Subscription,Active
1,CON_000002,ACC_00267,OPP_000008,2026-02-10,"12,863.89",2026-02-20,2027-02-15,Subscription,Active
2,CON_000003,ACC_00689,OPP_000010,2024-04-27,"2,409.55",2024-04-30,2025-04-25,Advertising,Active



── Billing ──


,customer_id,invoice_id,contract_id,billing_date,billing_amount,payment_status,currency
0,ACC_00418,INV_0000001,CON_000001,2025-02-18,"1,637.88",Paid,USD
1,ACC_00418,INV_0000002,CON_000001,2025-03-18,"1,637.88",Paid,GBP
2,ACC_00418,INV_0000003,CON_000001,2025-04-18,"1,637.88",Paid,USD


---
## 2. Null / Missing Value Audit
Check key identifier and metric fields across all tables.

In [4]:
key_fields = {
    'accounts':      ['account_id', 'segment', 'revenue_model'],
    'leads':         ['lead_id', 'account_id', 'lifecycle_stage', 'channel'],
    'opportunities': ['opportunity_id', 'account_id', 'opportunity_stage', 'amount'],
    'contracts':     ['contract_id', 'account_id', 'opportunity_id', 'contract_value'],
    'billing':       ['invoice_id', 'customer_id', 'contract_id', 'billing_amount'],
    'customer_success': ['account_id', 'customer_health_score', 'renewal_status'],
}

dfs = {
    'accounts': accounts, 'leads': leads, 'opportunities': opps,
    'contracts': contracts, 'billing': billing, 'customer_success': cs,
}

rows = []
for table, fields in key_fields.items():
    df = dfs[table]
    for f in fields:
        if f in df.columns:
            n_null = df[f].isna().sum()
            rows.append({
                'Table': table, 'Field': f,
                'Null Count': n_null,
                'Null %': f'{n_null / len(df) * 100:.2f}%',
                'Status': '⚠ Missing' if n_null > 0 else '✓ OK'
            })

null_df = pd.DataFrame(rows)
null_df[null_df['Null Count'] > 0].style.hide(axis='index')

Table,Field,Null Count,Null %,Status
leads,account_id,12,0.38%,⚠ Missing


---
## 3. Data Quality Issues
Intentional imperfections baked into the data — each one requires specific handling in the dbt staging layer.

### 3a. Duplicate Account IDs
Same `account_id` appears multiple times — needs deduplication in `stg_salesforce_accounts`.

In [5]:
dup_accounts = accounts[accounts.duplicated('account_id', keep=False)]
print(f'Rows with a duplicate account_id : {len(dup_accounts):,}')
print(f'Unique duplicated IDs            : {dup_accounts["account_id"].nunique():,}')
print()

# Show an example duplicate pair
example_id = dup_accounts['account_id'].iloc[0]
display(accounts[accounts['account_id'] == example_id][['account_id', 'account_name', 'segment', 'account_status']])

Rows with a duplicate account_id : 24
Unique duplicated IDs            : 12



,account_id,account_name,segment,account_status
44,ACC_00045,Pioneer Partners 0045,Enterprise,Prospect
1211,ACC_00045,Pioneer Partners 0045 Duplicate,Enterprise,Prospect


### 3b. Orphan Opportunities (no valid parent account)

In [6]:
valid_acc_ids = accounts.drop_duplicates('account_id')['account_id']
orphan_opps  = opps[~opps['account_id'].isin(valid_acc_ids)]

print(f'Orphan opportunities : {len(orphan_opps):,}  ({len(orphan_opps)/len(opps)*100:.1f}% of all opps)')
display(orphan_opps[['opportunity_id', 'account_id', 'opportunity_stage', 'amount']].head())

Orphan opportunities : 9  (0.5% of all opps)


,opportunity_id,account_id,opportunity_stage,amount
481,OPP_000482,ACC_ORPHAN_5,Closed-Lost,"8,048.21"
632,OPP_000633,ACC_ORPHAN_8,Closed-Won,"3,980.09"
804,OPP_000805,ACC_ORPHAN_7,Closed-Won,"14,373.35"
920,OPP_000921,ACC_ORPHAN_0,Proposal,"14,328.63"
1114,OPP_001115,ACC_ORPHAN_4,Negotiation,"2,500.00"


### 3c. Duplicate Contract IDs with Different Values
Same `contract_id` exists twice with slightly different `contract_value` — the core revenue reconciliation problem.

In [7]:
dup_contracts = contracts[contracts.duplicated('contract_id', keep=False)]
print(f'Rows with duplicate contract_id : {len(dup_contracts):,}')
print(f'Unique duplicated IDs           : {dup_contracts["contract_id"].nunique():,}\n')

# Show value discrepancy
discrepancy = (
    dup_contracts
    .groupby('contract_id')['contract_value']
    .agg(['min', 'max'])
    .assign(discrepancy=lambda d: (d['max'] - d['min']).round(2))
    .rename(columns={'min': 'value_v1', 'max': 'value_v2'})
)
display(discrepancy)

Rows with duplicate contract_id : 8
Unique duplicated IDs           : 4



,value_v1,value_v2,discrepancy
contract_id,,,
CON_000055,"8,470.67","8,604.10",133.43
CON_000103,"477,752.54","478,340.25",587.71
CON_000151,"42,472.89","43,168.41",695.52
CON_000355,"3,062.86","3,103.81",40.95


### 3d. Billing: Payment Status & Multi-Currency

In [8]:
payment_summary = (
    billing
    .groupby('payment_status')
    .agg(
        invoice_count=('invoice_id', 'count'),
        total_amount=('billing_amount', 'sum')
    )
    .assign(
        pct_invoices=lambda d: (d['invoice_count'] / len(billing) * 100).round(1),
        pct_amount=lambda d: (d['total_amount'] / billing['billing_amount'].sum() * 100).round(1)
    )
)
display(payment_summary)

print()
display(billing['currency'].value_counts().rename('invoice_count').to_frame())

,invoice_count,total_amount,pct_invoices,pct_amount
payment_status,,,,
Failed,61,"332,013.84",3.50,3.10
Paid,1525,"9,226,143.35",88.40,85.50
Pending,139,"1,238,534.25",8.10,11.50


,invoice_count
currency,
USD,1053
EUR,344
GBP,328


### 3e. Opportunity Close-Date Changes

In [9]:
cdc = (
    opps['close_date_changes']
    .value_counts()
    .sort_index()
    .rename('opportunity_count')
    .to_frame()
    .assign(pct=lambda d: (d['opportunity_count'] / len(opps) * 100).round(1))
)
cdc.index.name = 'times_close_date_changed'
display(cdc)

,opportunity_count,pct
times_close_date_changed,,
0,1097,60.90
1,464,25.80
2,175,9.70
3,64,3.60


---
## 4. Customer Lifecycle Join Chain
How well do the 7 source tables connect end-to-end?

```
Accounts → Opportunities → Contracts → Billing
                                          ↓
                                    Customer Success
```

In [10]:
acc_master   = accounts.drop_duplicates('account_id')
valid_acc    = set(acc_master['account_id'])
won_opps     = opps[opps['opportunity_stage'] == 'Closed-Won']
con_deduped  = contracts.drop_duplicates('contract_id')

accs_with_opp   = opps[opps['account_id'].isin(valid_acc)]['account_id'].nunique()
opps_with_con   = con_deduped[con_deduped['opportunity_id'].isin(won_opps['opportunity_id'])]['opportunity_id'].nunique()
cons_with_bil   = billing['contract_id'].isin(con_deduped['contract_id']).sum()
cons_no_billing = con_deduped[~con_deduped['contract_id'].isin(billing['contract_id'])]

chain = pd.DataFrame([
    {'Step': 'Accounts (unique)',          'Count': len(valid_acc),     'Coverage': '—'},
    {'Step': '→ with ≥1 Opportunity',      'Count': accs_with_opp,     'Coverage': f'{accs_with_opp/len(valid_acc)*100:.1f}%'},
    {'Step': 'Opportunities (total)',       'Count': len(opps),         'Coverage': '—'},
    {'Step': '→ Closed-Won',               'Count': len(won_opps),     'Coverage': f'{len(won_opps)/len(opps)*100:.1f}%'},
    {'Step': 'Contracts (unique)',          'Count': len(con_deduped),  'Coverage': '—'},
    {'Step': '→ Linked to valid Opp',      'Count': opps_with_con,     'Coverage': f'{opps_with_con/len(con_deduped)*100:.1f}%'},
    {'Step': 'Billing invoices',            'Count': len(billing),      'Coverage': '—'},
    {'Step': '→ Linked to valid Contract', 'Count': cons_with_bil,     'Coverage': f'{cons_with_bil/len(billing)*100:.1f}%'},
    {'Step': 'Customer Success records',   'Count': len(cs),           'Coverage': '—'},
    {'Step': '→ Matched to valid Account', 'Count': cs["account_id"].isin(valid_acc).sum(), 'Coverage': '100.0%'},
])

display(chain.style.hide(axis='index'))

print(f'\n⚠  Contracts with NO billing rows : {len(cons_no_billing):,}')
display(cons_no_billing[['contract_id', 'account_id', 'contract_value', 'revenue_model', 'contract_status']].head())

Step,Count,Coverage
Accounts (unique),1200,—
→ with ≥1 Opportunity,922,76.8%
Opportunities (total),1800,—
→ Closed-Won,373,20.7%
Contracts (unique),373,—
→ Linked to valid Opp,373,100.0%
Billing invoices,1725,—
→ Linked to valid Contract,1725,100.0%
Customer Success records,697,—
→ Matched to valid Account,697,100.0%



⚠  Contracts with NO billing rows : 13


,contract_id,account_id,contract_value,revenue_model,contract_status
1,CON_000002,ACC_00267,"12,863.89",Subscription,Active
21,CON_000022,ACC_00777,"50,867.70",Subscription,Active
95,CON_000096,ACC_00420,"52,034.05",Transactional,Active
107,CON_000108,ACC_00958,"67,053.72",Advertising,Active
147,CON_000148,ACC_01136,"15,570.48",Transactional,Active


---
## 5. GTM Funnel — Lead to Revenue
End-to-end conversion from first lead to closed-won deal.

In [11]:
# Lead funnel
total_leads = len(leads)
mqls  = leads['lifecycle_stage'].isin(['MQL', 'SQL', 'Converted']).sum()
sqls  = leads['lifecycle_stage'].isin(['SQL', 'Converted']).sum()
convd = (leads['lifecycle_stage'] == 'Converted').sum()

# Opportunity funnel
won  = (opps['opportunity_stage'] == 'Closed-Won').sum()
lost = (opps['opportunity_stage'] == 'Closed-Lost').sum()

funnel = pd.DataFrame([
    {'Stage': 'Total Leads',       'Count': total_leads, 'Conversion': '—'},
    {'Stage': '→ MQL',             'Count': mqls,  'Conversion': f'{mqls/total_leads*100:.1f}%'},
    {'Stage': '→ SQL',             'Count': sqls,  'Conversion': f'{sqls/total_leads*100:.1f}%'},
    {'Stage': '→ Converted',       'Count': convd, 'Conversion': f'{convd/total_leads*100:.1f}%'},
    {'Stage': '─────────────────', 'Count': '',    'Conversion': ''},
    {'Stage': 'Total Opps',        'Count': len(opps), 'Conversion': '—'},
    {'Stage': '→ Open / Active',   'Count': len(opps)-won-lost, 'Conversion': f'{(len(opps)-won-lost)/len(opps)*100:.1f}%'},
    {'Stage': '→ Closed-Won',      'Count': won,  'Conversion': f'{won/len(opps)*100:.1f}%'},
    {'Stage': '→ Closed-Lost',     'Count': lost, 'Conversion': f'{lost/len(opps)*100:.1f}%'},
    {'Stage': 'Win Rate (W/Closed)','Count': '',   'Conversion': f'{won/(won+lost)*100:.1f}%'},
])

display(funnel.style.hide(axis='index'))

Stage,Count,Conversion
Total Leads,3200,—
→ MQL,1852,57.9%
→ SQL,954,29.8%
→ Converted,383,12.0%
─────────────────,,
Total Opps,1800,—
→ Open / Active,1147,63.7%
→ Closed-Won,373,20.7%
→ Closed-Lost,280,15.6%
Win Rate (W/Closed),,57.1%


### Win Rate & Average Deal Size by Segment

In [12]:
seg_summary = (
    opps
    .groupby('segment')
    .apply(lambda g: pd.Series({
        'total_opps':   len(g),
        'won':          (g['opportunity_stage'] == 'Closed-Won').sum(),
        'lost':         (g['opportunity_stage'] == 'Closed-Lost').sum(),
        'avg_deal_size':g[g['opportunity_stage'] == 'Closed-Won']['amount'].mean(),
        'total_bookings': g[g['opportunity_stage'] == 'Closed-Won']['amount'].sum(),
    }), include_groups=False)
    .assign(win_rate=lambda d: (d['won'] / (d['won'] + d['lost']) * 100).round(1))
    [['total_opps', 'won', 'lost', 'win_rate', 'avg_deal_size', 'total_bookings']]
)

display(seg_summary.style.format({
    'win_rate': '{:.1f}%',
    'avg_deal_size': '${:,.0f}',
    'total_bookings': '${:,.0f}',
}))

,total_opps,won,lost,win_rate,avg_deal_size,total_bookings
segment,,,,,,
Enterprise,337.000000,75.000000,52.000000,59.1%,"$73,780","$5,533,522"
Mid-Market,618.000000,140.000000,102.000000,57.9%,"$29,392","$4,114,822"
SMB,845.000000,158.000000,126.000000,55.6%,"$14,058","$2,221,128"


### Lead Volume & SQL Conversion Rate by Channel

In [13]:
ch_summary = (
    leads
    .groupby('channel')
    .apply(lambda g: pd.Series({
        'lead_count': len(g),
        'mql_count':  g['lifecycle_stage'].isin(['MQL','SQL','Converted']).sum(),
        'sql_count':  g['lifecycle_stage'].isin(['SQL','Converted']).sum(),
    }), include_groups=False)
    .assign(
        mql_rate=lambda d: (d['mql_count'] / d['lead_count'] * 100).round(1),
        sql_rate=lambda d: (d['sql_count'] / d['lead_count'] * 100).round(1),
    )
    .sort_values('lead_count', ascending=False)
)

display(ch_summary.style.format({
    'mql_rate': '{:.1f}%',
    'sql_rate': '{:.1f}%',
}))

,lead_count,mql_count,sql_count,mql_rate,sql_rate
channel,,,,,
Organic,704,419,214,59.5%,30.4%
Paid Search,570,339,173,59.5%,30.4%
Outbound,447,246,136,55.0%,30.4%
Paid Social,442,263,138,59.5%,31.2%
Partner,385,231,121,60.0%,31.4%
Referral,365,205,99,56.2%,27.1%
Events,287,149,73,51.9%,25.4%


---
## 6. Pipeline Snapshot
Current open pipeline — size, stage distribution, and at-risk signals.

In [14]:
open_opps = opps[~opps['opportunity_stage'].isin(['Closed-Won', 'Closed-Lost'])].copy()

print(f'Open opportunities : {len(open_opps):,}')
print(f'Total pipeline     : ${open_opps["amount"].sum():,.0f}')
print(f'Average deal size  : ${open_opps["amount"].mean():,.0f}\n')

pipeline_by_stage = (
    open_opps
    .groupby('opportunity_stage')
    .agg(
        opp_count=('opportunity_id', 'count'),
        pipeline_value=('amount', 'sum'),
        avg_deal=('amount', 'mean'),
    )
    .reindex(['Prospecting', 'Qualification', 'Proposal', 'Negotiation'])
    .assign(pct_of_pipeline=lambda d: (d['pipeline_value'] / d['pipeline_value'].sum() * 100).round(1))
)

display(pipeline_by_stage.style.format({
    'pipeline_value': '${:,.0f}',
    'avg_deal': '${:,.0f}',
    'pct_of_pipeline': '{:.1f}%',
}))

Open opportunities : 1,147
Total pipeline     : $32,872,097
Average deal size  : $28,659



,opp_count,pipeline_value,avg_deal,pct_of_pipeline
opportunity_stage,,,,
Prospecting,259,"$7,265,053","$28,050",22.1%
Qualification,328,"$8,726,894","$26,606",26.5%
Proposal,323,"$9,489,807","$29,380",28.9%
Negotiation,237,"$7,390,344","$31,183",22.5%


### At-Risk Signals
Three signals that indicate an opportunity may not close as planned.

In [15]:
today = pd.Timestamp('2025-12-31')  # use data end date as reference

past_due      = open_opps[open_opps['expected_close_date'] < today]
multi_changes = open_opps[open_opps['close_date_changes'] >= 2]
stale         = open_opps[(today - open_opps['last_activity_at']).dt.days > 60]

risk_summary = pd.DataFrame([
    {'Risk Signal': 'Past expected close date',   'Count': len(past_due),      'Pct of Open Opps': f'{len(past_due)/len(open_opps)*100:.1f}%', 'Pipeline at Risk': f'${past_due["amount"].sum():,.0f}'},
    {'Risk Signal': 'Close date changed ≥2x',     'Count': len(multi_changes), 'Pct of Open Opps': f'{len(multi_changes)/len(open_opps)*100:.1f}%', 'Pipeline at Risk': f'${multi_changes["amount"].sum():,.0f}'},
    {'Risk Signal': 'No activity in 60+ days',    'Count': len(stale),         'Pct of Open Opps': f'{len(stale)/len(open_opps)*100:.1f}%', 'Pipeline at Risk': f'${stale["amount"].sum():,.0f}'},
])

display(risk_summary.style.hide(axis='index'))

Risk Signal,Count,Pct of Open Opps,Pipeline at Risk
Past expected close date,1109,96.7%,"$31,972,432"
Close date changed ≥2x,136,11.9%,"$3,987,279"
No activity in 60+ days,1060,92.4%,"$31,127,815"


---
## 7. Revenue Reconciliation
Why do CRM bookings ≠ contract value ≠ billed amount ≠ collected?

In [16]:
won_opps      = opps[opps['opportunity_stage'] == 'Closed-Won']
con_deduped   = contracts.drop_duplicates('contract_id')

crm_bookings      = won_opps['amount'].sum()
contract_value    = con_deduped['contract_value'].sum()
billed_total      = billing['billing_amount'].sum()
paid_total        = billing[billing['payment_status'] == 'Paid']['billing_amount'].sum()
pending_total     = billing[billing['payment_status'] == 'Pending']['billing_amount'].sum()
failed_total      = billing[billing['payment_status'] == 'Failed']['billing_amount'].sum()

recon = pd.DataFrame([
    {'Revenue Layer': 'CRM Bookings (Closed-Won)',   'Amount': crm_bookings,    'vs Previous': '—',                              'Note': 'Opp amount at deal close'},
    {'Revenue Layer': 'Contracts (signed value)',    'Amount': contract_value,  'vs Previous': f'${contract_value-crm_bookings:+,.0f}',   'Note': 'Contract renegotiation / adjustments'},
    {'Revenue Layer': 'Billed (all invoices)',       'Amount': billed_total,    'vs Previous': f'${billed_total-contract_value:+,.0f}',   'Note': 'Timing: not all contracts billed yet'},
    {'Revenue Layer': '  → Paid',                   'Amount': paid_total,      'vs Previous': f'${paid_total-billed_total:+,.0f}',       'Note': 'Collected revenue'},
    {'Revenue Layer': '  → Pending',                'Amount': pending_total,   'vs Previous': '—',                              'Note': 'At risk of failure'},
    {'Revenue Layer': '  → Failed',                 'Amount': failed_total,    'vs Previous': '—',                              'Note': 'Revenue leakage'},
])

display(recon.style.format({'Amount': '${:,.0f}'}).hide(axis='index'))

print(f'\nCollected / Contracted ratio : {paid_total/contract_value*100:.1f}%')
print(f'Revenue leakage (failed)     : ${failed_total:,.0f}  ({failed_total/contract_value*100:.1f}% of contracts)')
print(f'Revenue at risk (pending)    : ${pending_total:,.0f}  ({pending_total/contract_value*100:.1f}% of contracts)')

Revenue Layer,Amount,vs Previous,Note
CRM Bookings (Closed-Won),"$11,869,473",—,Opp amount at deal close
Contracts (signed value),"$11,818,603","$-50,870",Contract renegotiation / adjustments
Billed (all invoices),"$10,796,691","$-1,021,911",Timing: not all contracts billed yet
→ Paid,"$9,226,143","$-1,570,548",Collected revenue
→ Pending,"$1,238,534",—,At risk of failure
→ Failed,"$332,014",—,Revenue leakage



Collected / Contracted ratio : 78.1%
Revenue leakage (failed)     : $332,014  (2.8% of contracts)
Revenue at risk (pending)    : $1,238,534  (10.5% of contracts)


---
## 8. Customer Health & Retention
Renewal status, churn reasons, and expansion opportunity.

In [17]:
health_summary = (
    cs
    .groupby('renewal_status')
    .agg(
        account_count=('account_id', 'count'),
        avg_health=('customer_health_score', 'mean'),
        total_expansion=('expansion_amount', 'sum'),
    )
    .assign(pct=lambda d: (d['account_count'] / len(cs) * 100).round(1))
    .sort_values('account_count', ascending=False)
)

display(health_summary.style.format({
    'avg_health': '{:.1f}',
    'total_expansion': '${:,.0f}',
    'pct': '{:.1f}%',
}))

,account_count,avg_health,total_expansion,pct
renewal_status,,,,
Healthy,453,78.3,"$640,104",65.0%
Needs Attention,116,50.4,$0,16.6%
Churned,113,47.9,$0,16.2%
At Risk,15,27.8,$0,2.2%


In [18]:
print(f'Avg health score (all customers) : {cs["customer_health_score"].mean():.1f}')
print(f'Accounts with expansion revenue  : {(cs["expansion_amount"] > 0).sum():,}')
print(f'Total expansion ARR              : ${cs["expansion_amount"].sum():,.0f}\n')

# Churn reasons
print('Churn Reasons (churned accounts only):')
churned = cs[cs['churn_reason'].notna()]
churn_reasons = (
    churned['churn_reason']
    .value_counts()
    .rename('count')
    .to_frame()
    .assign(pct=lambda d: (d['count'] / len(churned) * 100).round(1))
)
display(churn_reasons.style.format({'pct': '{:.1f}%'}))

Avg health score (all customers) : 67.6
Accounts with expansion revenue  : 332
Total expansion ARR              : $640,104

Churn Reasons (churned accounts only):


,count,pct
churn_reason,,
Poor Fit,27,23.9%
Budget,25,22.1%
Competition,25,22.1%
Low Adoption,21,18.6%
Business Closure,15,13.3%


---
## Summary: What This Means for dbt Modeling

| Finding | dbt Model to Handle It |
|---------|------------------------|
| Duplicate account IDs (12 pairs) | `stg_salesforce_accounts` — `ROW_NUMBER()` dedup |
| 12 leads with missing `account_id` | `stg_marketing_leads` — null flag + exclusion |
| 9 orphan opportunities | `int_gtm_funnel` — left join with null handling |
| 4 duplicate contract IDs with different values | `stg_contracts` — flag + alert, pick canonical |
| 13 contracts with no billing | `int_contract_to_revenue` — left join, flag as gap |
| $332K failed payments | `fct_billing` — `payment_status` filter |
| Multi-currency billing | `stg_billing` — currency normalization |
| 96% of open opps past close date | `fct_pipeline` — `days_past_due` calculated field |

**Next step → Step 2: dbt staging models**